In [1]:
%config IPCompleter.greedy=True
%config IPCompleter.use_jedi=False
from cytools import Polytope, Cone
from cytools.vector_config.fan import Fan
from cytools.vector_config import VectorConfiguration,Fan
import numpy as np
from itertools import combinations_with_replacement,combinations
# from cytools.config import enable_experimental_features
# enable_experimental_features()
np.set_printoptions(linewidth=2000)
from cygv import compute_gv
import sys
sys.path.append(r"../..")
from CYorientifolds_FTheoryUplifts.FT_CY_local import F_Theory_Uplift
from CYorientifolds_FTheoryUplifts import Uplift_functions_local as UF
# sys.path.append(r"/home/hassfeld/Documents/Bjoern/Bachelorarbeit/CYtools/")
# from Local_databases import fetch_F_Theory_uplifts_local 
np.set_printoptions(legacy="1.25")

In [ ]:
p=Polytope([[-1,0,0,0],[1,1,0,0],[1,0,1,0],[1,0,0,1],[1,-1,-1,-1]]).dual()
F=F_Theory_Uplift(p)
fan=VectorConfiguration(F.pol_M_conv().points()[1:]).triangulate()
LB1=np.zeros(len(fan.vectors()),dtype=int)
LB1[np.array(F.nef_partition_M(as_divisors=True)[0])-1]=1
LB2=1-LB1.copy()
Q=UF.symmetric_glsm_uplift(F)[0]
IN=fan.intersection_numbers()
INQ=UF.tensor_symm_in_basis(F.intersection_numbers_M(),F.basis_homology_M(),Q)
np1,np2=F.nef_partition_M(zero_based=False)
np1=np.array(np1)
np2=np.array(np2)
four_cones=UF.get_lower_dimensional_cones(fan.cones(),4)
two_cones=UF.get_lower_dimensional_cones(fan.cones(),2)
int_mat=np.zeros((len(two_cones),len(four_cones)))
for i,c2 in enumerate(two_cones):
    for j,c4 in enumerate(four_cones):
        t=tuple(sorted(c2+c4))
        int_mat[i,j]=IN.get(t,0)
m=np.rint(int_mat).astype(int)
combs=[sorted((a,b)) for a in np1 for b in np2]
r=np.linalg.matrix_rank(m)
b8,t8=UF.hnf(m,return_transform=True)
b4,t4=UF.hnf(m.T,return_transform=True)
basis8_cycles=b8[:r]
basis4_cycles=b4[:r]
t8inv=np.rint(np.linalg.inv(t8)).astype(int)
t4inv=np.rint(np.linalg.inv(t4)).astype(int)
assert np.array_equal(b8, t8 @ m)
assert np.array_equal(b4, t4 @ m.T)
int_form=(t8@m@t4.T)[:r,:r]
# LB1,LB2=F.nef_partition_M(as_divisors=True)
C8 = t8[:r, :]
C4 = t4[:r, :]
# their intersection vectors
B8 = b8[:r, :]
B4 = b4[:r, :]
G=C8@m@C4.T
Ginv=np.rint(np.linalg.inv(G)).astype(int)
M=Ginv.T@C4
N= Ginv@C8
basis_map,intform,surface_map,gram=UF.intersection_form_H22_vertical(INQ)
c2=UF.tensor_symm_in_basis(UF.toric_chern_class(F.M_conv_toric_fan(),F.nef_partition_M(as_divisors=True,toric_fan_adapted=True),2),F.basis_homology_M(),Q)
c2_vec=np.zeros(basis_map.shape[1])
for i in range(len(c2_vec)):
    c2_vec[i]=c2.get(surface_map.get(i),0)

In [30]:
from fractions import Fraction
import numpy as np

def cartier_data(D, V, max_cones):
    data = []

    for cone in max_cones:
        I = np.asarray(cone, dtype=int) - 1

        A = V[I]                 # NO transpose
        rhs = -np.asarray(D)[I]

        m0 = np.linalg.solve(A, rhs)
        m = np.rint(m0).astype(int)

        if not np.array_equal(A @ m, rhs):
            raise ValueError("Cartier solve was not exact")

        data.append(m)

    return np.array(data, dtype=int)


def semiample_dimension(D, V, max_cones):
    ms = cartier_data(D, V, max_cones)
    diffs = ms[1:] - ms[0]
    return np.linalg.matrix_rank(diffs), ms


V = np.asarray(fan.vectors(), dtype=int)
max_cones = list(fan.cones())

k1, ms1 = semiample_dimension(LB1, V, max_cones)
k2, ms2 = semiample_dimension(LB2, V, max_cones)

print(k1, k2)


# def semiample_dimension(D, V, max_cones):
#     ms = cartier_data(D, V, max_cones)
#     diffs = ms[1:] - ms[0]

#     # for these small integer matrices numerical rank should be checked
#     # against an exact HNF/rank routine if available
#     return np.linalg.matrix_rank(diffs), ms


# k1, ms1 = semiample_dimension(LB1, np.array(fan.vectors()), fan.cones())
# k2, ms2 = semiample_dimension(LB2, np.array(fan.vectors()), fan.cones())

# print(k1, k2)

4 6


In [36]:
K = cy_cycles @ m.T

hK, tK = UF.hnf(K, return_transform=True)
nzK = np.flatnonzero(np.any(hK != 0, axis=1))

Qcy = tK[nzK, :]          # 6 x 98
Gcy = Qcy @ K @ Qcy.T     # 6 x 6
from functools import lru_cache
import numpy as np

L1 = np.asarray(LB1, dtype=int)
L2 = np.asarray(LB2, dtype=int)
nrays = len(V)

@lru_cache(None)
def int_X4(mon4):
    """
    Integral over X of four toric divisors.
    mon4 uses your 1-based ray labels.
    """
    mon4 = tuple(mon4)

    val = 0
    for r, ar in enumerate(L1, start=1):
        if ar == 0:
            continue

        for s, bs in enumerate(L2, start=1):
            if bs == 0:
                continue

            key = tuple(sorted(mon4 + (r, s)))
            val += ar * bs * IN.get(key, 0)

    return int(val)
Sraw = np.zeros((len(two_cones), nrays), dtype=np.uint8)

for i, c2 in enumerate(two_cones):
    a, b = c2

    # a and b should be different for a genuine 2-cone
    assert a != b

    for c in range(1, nrays + 1):

        # Sq^2(D_a D_b)
        # = D_a^2 D_b + D_a D_b^2   mod 2

        x = int_X4((a, a, b, c))
        y = int_X4((a, b, b, c))

        Sraw[i, c - 1] = (x + y) % 2
S = ((Qcy % 2) @ Sraw) % 2

print(S)
def gf2_rref(A):
    A = np.asarray(A, dtype=np.uint8).copy() % 2

    m, n = A.shape
    pivots = []
    r = 0

    for c in range(n):
        inds = np.flatnonzero(A[r:, c])
        if len(inds) == 0:
            continue

        p = r + inds[0]
        A[[r, p]] = A[[p, r]]

        for i in range(m):
            if i != r and A[i, c]:
                A[i] ^= A[r]

        pivots.append(c)
        r += 1

        if r == m:
            break

    return A, pivots


def gf2_rank(A):
    _, pivots = gf2_rref(A)
    return len(pivots)


def gf2_nullspace(A):
    """
    Rows form a basis of ker(A) over F_2.
    """
    R, pivots = gf2_rref(A)

    n = A.shape[1]
    free = [j for j in range(n) if j not in pivots]

    basis = []

    for f in free:
        x = np.zeros(n, dtype=np.uint8)
        x[f] = 1

        for row, p in enumerate(pivots):
            x[p] = R[row, f]

        basis.append(x)

    if not basis:
        return np.zeros((0, n), dtype=np.uint8)

    return np.array(basis, dtype=np.uint8)
G2 = np.asarray(Gcy, dtype=np.int64) % 2

# coefficient row a must satisfy a @ G2 = 0
Rad = gf2_nullspace(G2.T)

print("dim radical =", Rad.shape[0])
print(Rad)
D = (Rad @ S) % 2

print("Steenrod detector:")
print(D)

print("rank =", gf2_rank(D))
print("radical dimension =", Rad.shape[0])

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]]
dim radical = 4
[[1 0 0 0 0 0]
 [0 1 1 0 0 0]
 [0 0 0 0 1 0]
 [0 1 0 0 0 1]]
Steenrod detector:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
rank = 0
radical dimension = 4


In [37]:
import numpy as np

# ============================================================
# 0. Basic CY lattice from your existing data
#    Requires:
#      cy_cycles, m, two_cones, LB1, LB2, UF
# ============================================================

K = cy_cycles @ m.T

hK, tK = UF.hnf(K, return_transform=True)
nzK = np.flatnonzero(np.any(hK != 0, axis=1))

Qcy = tK[nzK, :]                  # rank-6 basis in terms of raw 2-cones
Gcy = Qcy @ K @ Qcy.T             # Gram matrix in this basis

r = Gcy.shape[0]
assert r == 6

print("Gcy =")
print(Gcy)
print("\nSNF(Gcy) =")
print(UF.snf(Gcy))


# ============================================================
# 1. Linear algebra over F_2
# ============================================================

def gf2_rref(A):
    A = np.asarray(A, dtype=np.uint8).copy() % 2
    m_, n_ = A.shape

    pivots = []
    row = 0

    for col in range(n_):
        inds = np.flatnonzero(A[row:, col])
        if len(inds) == 0:
            continue

        p = row + inds[0]
        A[[row, p]] = A[[p, row]]

        for i in range(m_):
            if i != row and A[i, col]:
                A[i] ^= A[row]

        pivots.append(col)
        row += 1

        if row == m_:
            break

    return A, pivots


def gf2_nullspace(A):
    """
    Returns a matrix whose ROWS form a basis of ker(A) over F_2.
    """
    A = np.asarray(A, dtype=np.uint8) % 2
    R, pivots = gf2_rref(A)

    n_ = A.shape[1]
    free = [j for j in range(n_) if j not in pivots]

    basis = []

    for f in free:
        x = np.zeros(n_, dtype=np.uint8)
        x[f] = 1

        for row, p in enumerate(pivots):
            x[p] = R[row, f]

        basis.append(x)

    if len(basis) == 0:
        return np.zeros((0, n_), dtype=np.uint8)

    return np.array(basis, dtype=np.uint8)


# Coefficient row a is in radical iff a @ Gcy = 0 mod 2
Rad = gf2_nullspace((Gcy % 2).T)

print("\nRadical basis mod 2:")
print(Rad)
print("dim radical =", len(Rad))


# ============================================================
# 2. Construct c2(X)
#
# For X = D1 cap D2 Calabi-Yau:
#
#   c2(X) = (c2(A) - D1*D2)|_X
#
# For smooth toric A:
#
#   c2(A) = sum_{rho<sigma} D_rho D_sigma
#
# two_cones contains the nonzero toric products D_a D_b.
# ============================================================

L1 = np.asarray(LB1, dtype=int)
L2 = np.asarray(LB2, dtype=int)

c2raw = np.zeros(len(two_cones), dtype=int)

for j, cone in enumerate(two_cones):
    a, b = cone
    ia = a - 1
    ib = b - 1

    # coefficient from c2(A) - D1*D2
    #
    # D1*D2 contributes
    # L1[a]*L2[b] + L1[b]*L2[a]
    c2raw[j] = (
        1
        - L1[ia] * L2[ib]
        - L1[ib] * L2[ia]
    )


# Pairing of each rank-6 basis element with c2(X)
c2pair = Qcy @ K @ c2raw

print("\n<e_i, c2(X)> =")
print(c2pair)


# ============================================================
# 3. Essential consistency check:
#
# c2(X) is characteristic on a CY4:
#
#       e_i^2 = e_i.c2(X) mod 2
# ============================================================

diag2 = np.diag(Gcy) % 2
c2mod2 = c2pair % 2

print("\ndiag(Gcy) mod 2 =", diag2)
print("e_i.c2   mod 2 =", c2mod2)

if not np.array_equal(diag2, c2mod2):
    raise ValueError(
        "Characteristic-vector check failed: "
        "diag(G) != pairing with c2(X) mod 2. "
        "Check c2raw / cone conventions before proceeding."
    )

print("Characteristic-vector check: PASSED")


# ============================================================
# 4. Test every possible order-2 extension
#
# If y = x/2 is an integral class, necessary conditions are:
#
#   (i)  x.G is even
#
#   (ii) x.c2 is even
#
#   (iii) y^2 = y.c2 mod 2
#
# which becomes
#
#       x^2 - 2*x.c2 = 0 mod 8.
#
# Any nontrivial 2-primary saturation must contain an
# order-2 element, so if NONE survive, L is saturated.
# ============================================================

survivors = []

print("\nCandidate half-classes:")
print("-" * 75)

for mask in range(1, 1 << len(Rad)):

    # coefficient vector x in the six-dimensional basis, mod 2
    a = np.zeros(r, dtype=np.uint8)

    for j in range(len(Rad)):
        if (mask >> j) & 1:
            a ^= Rad[j]

    aZ = a.astype(int)

    pairings = aZ @ Gcy
    x2 = int(aZ @ Gcy @ aZ)
    xc2 = int(aZ @ c2pair)

    radical_ok = np.all(pairings % 2 == 0)
    c2_even = (xc2 % 2 == 0)
    wu_ok = ((x2 - 2 * xc2) % 8 == 0)

    possible = radical_ok and c2_even and wu_ok

    print(
        f"a = {aZ}   "
        f"x^2 = {x2:4d}   "
        f"x.c2 = {xc2:4d}   "
        f"survives = {possible}"
    )

    if possible:
        survivors.append(aZ.copy())


# ============================================================
# 5. Result
# ============================================================

print("\n" + "=" * 75)
print("NUMBER OF SURVIVING HALF-CLASS DIRECTIONS:", len(survivors))

if len(survivors) == 0:
    print("""
RESULT:
No nonzero order-2 extension is possible.

Since L^vee/L is 2-primary, any nontrivial finite saturation V/L
would contain an element of order 2.

Therefore:

    H^{2,2}_vert(X,Z) = L

and your current rank-6 lattice is already saturated.
""")

else:
    print("\nSurviving coefficient vectors a:")
    for a in survivors:
        print(a, "   -> candidate class  (a_1 e_1 + ... + a_6 e_6)/2")

    print("""
RESULT:
These are NOT yet proven integral classes.

They are exactly the half-integral directions that are not excluded
by:
    - integrality of the intersection pairing,
    - the mod-2 radical condition,
    - the characteristic class c2(X),
    - the Wu congruence.

Further topology is required only for these surviving directions.
""")

Gcy =
[[ 8  4  8  2  8 12]
 [ 4 -4  4  1  0  4]
 [ 8  4  4  1  8 12]
 [ 2  1  1  0  2  3]
 [ 8  0  8  2  0  8]
 [12  4 12  3  8 12]]

SNF(Gcy) =
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 4 0 0 0]
 [0 0 0 4 0 0]
 [0 0 0 0 8 0]
 [0 0 0 0 0 8]]

Radical basis mod 2:
[[1 0 0 0 0 0]
 [0 1 1 0 0 0]
 [0 0 0 0 1 0]
 [0 1 0 0 0 1]]
dim radical = 4

<e_i, c2(X)> =
[172 -10 182  48  44 162]

diag(Gcy) mod 2 = [0 0 0 0 0 0]
e_i.c2   mod 2 = [0 0 0 0 0 0]
Characteristic-vector check: PASSED

Candidate half-classes:
---------------------------------------------------------------------------
a = [1 0 0 0 0 0]   x^2 =    8   x.c2 =  172   survives = True
a = [0 1 1 0 0 0]   x^2 =    8   x.c2 =  172   survives = True
a = [1 1 1 0 0 0]   x^2 =   40   x.c2 =  344   survives = True
a = [0 0 0 0 1 0]   x^2 =    0   x.c2 =   44   survives = True
a = [1 0 0 0 1 0]   x^2 =   24   x.c2 =  216   survives = True
a = [0 1 1 0 1 0]   x^2 =   24   x.c2 =  216   survives = True
a = [1 1 1 0 1 0]   x^2 =   72   x.c2 =  388

In [38]:
import numpy as np

def gf2_rank(A):
    A = np.asarray(A, dtype=np.uint8).copy() % 2
    m, n = A.shape
    r = 0

    for c in range(n):
        p = next((i for i in range(r, m) if A[i, c]), None)
        if p is None:
            continue

        A[[r, p]] = A[[p, r]]

        for i in range(m):
            if i != r and A[i, c]:
                A[i] ^= A[r]

        r += 1
        if r == m:
            break

    return r

# P[i,a] = intersection of e_i with integral 4-cycle Gamma_a
print("rank mod 2 =", gf2_rank(P))

if gf2_rank(P) == 6:
    print("PROVED: current rank-6 lattice is saturated.")

NameError: name 'P' is not defined

In [39]:
import numpy as np
from fractions import Fraction

# ============================================================
# EXACT SATURATION OF THE VERTICAL LATTICE
#
# INPUT:
#   G  : r x r Gram matrix of your current vertical Z-lattice L
#
#   P  : r x N INTEGER matrix
#
#        P[i,a] = integral of e_i over Gamma_a
#
#        Gamma_a must generate the integral dual lattice sufficiently
#        to detect H^4(X,Z) on the rational vertical subspace.
#
#   UF.snf(A, return_transform=True) is assumed to satisfy
#
#        D = U @ A @ V
#
# OUTPUT:
#   T      rational change of basis, V_true = T * L
#   Gtrue  exact Gram matrix in the saturated basis
#   index  [V_true : L]
# ============================================================


def exact_vertical_saturation(G, P, UF):

    G = np.asarray(G, dtype=object)
    P = np.asarray(P, dtype=int)

    r = G.shape[0]

    assert G.shape == (r, r)
    assert P.shape[0] == r

    # --------------------------------------------------------
    # 1. Smith form of the ACTUAL integral embedding/pairing
    # --------------------------------------------------------

    D, U, V = UF.snf(P, return_transform=True)

    # adapt this assertion if UF uses another return convention
    assert np.array_equal(D, U @ P @ V)

    # nonzero Smith factors
    d = []

    for i in range(r):
        di = abs(int(D[i, i]))
        if di == 0:
            raise ValueError(
                "P does not detect all vertical rational directions: "
                f"rank < {r}"
            )
        d.append(di)

    print("Smith factors of actual integral embedding:")
    print(d)

    # --------------------------------------------------------
    # 2. Exact saturation index
    # --------------------------------------------------------

    index = int(np.prod(d, dtype=object))

    print("\nSaturation index [V:L] =", index)

    # --------------------------------------------------------
    # 3. Rational basis transformation
    #
    # T = diag(1/d_i) U
    # --------------------------------------------------------

    T = np.empty((r, r), dtype=object)

    for i in range(r):
        for j in range(r):
            T[i, j] = Fraction(int(U[i, j]), d[i])

    # --------------------------------------------------------
    # 4. Exact Gram matrix in new basis
    # --------------------------------------------------------

    Gfrac = np.empty((r, r), dtype=object)

    for i in range(r):
        for j in range(r):
            Gfrac[i, j] = Fraction(int(G[i, j]), 1)

    def matmul_frac(A, B):
        out = np.empty((A.shape[0], B.shape[1]), dtype=object)

        for i in range(A.shape[0]):
            for j in range(B.shape[1]):
                s = Fraction(0, 1)
                for k in range(A.shape[1]):
                    s += A[i, k] * B[k, j]
                out[i, j] = s

        return out

    Gtrue = matmul_frac(matmul_frac(T, Gfrac), T.T)

    # --------------------------------------------------------
    # 5. Genuine integral lattice => Gram must be integral
    # --------------------------------------------------------

    bad = []

    for i in range(r):
        for j in range(r):
            if Gtrue[i, j].denominator != 1:
                bad.append((i, j, Gtrue[i, j]))

    if bad:
        raise ValueError(
            "The proposed P cannot be the full integral pairing lattice: "
            "the resulting saturated Gram matrix is not integral.\n"
            f"First bad entries: {bad[:10]}"
        )

    Gtrue_int = np.array(
        [[int(Gtrue[i, j]) for j in range(r)] for i in range(r)],
        dtype=object
    )

    # --------------------------------------------------------
    # 6. Determinant consistency
    # --------------------------------------------------------

    det_old = round(np.linalg.det(np.asarray(G, dtype=float)))
    det_new_expected = Fraction(abs(det_old), index**2)

    print("|det G_old| =", abs(det_old))
    print("Expected |det G_true| =", det_new_expected)

    if det_new_expected.denominator != 1:
        raise ValueError(
            "Impossible saturation: index^2 does not divide |det G|."
        )

    det_new = round(
        np.linalg.det(np.asarray(Gtrue_int, dtype=float))
    )

    if abs(det_new) != int(det_new_expected):
        raise ValueError(
            f"Determinant mismatch: got {det_new}, "
            f"expected +/-{det_new_expected}"
        )

    # --------------------------------------------------------
    # 7. Smith form of final Gram lattice
    # --------------------------------------------------------

    Gtrue_np = np.asarray(Gtrue_int, dtype=np.int64)
    Strue = UF.snf(Gtrue_np)

    print("\nExact basis transformation T:")
    print(T)

    print("\nTrue integral Gram matrix:")
    print(Gtrue_np)

    print("\nSNF(true Gram):")
    print(Strue)

    return {
        "smith_embedding": d,
        "index": index,
        "T": T,
        "Gtrue": Gtrue_np,
        "snf_Gtrue": Strue,
    }

In [40]:
result = exact_vertical_saturation(Gcy, P, UF)

NameError: name 'P' is not defined

In [3]:
a,s,t=UF.snf(intform,return_decomposition=True)
ainv=np.linalg.inv(a)
s@intform@t

array([[1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 4, 0, 0, 0],
       [0, 0, 0, 4, 0, 0],
       [0, 0, 0, 0, 8, 0],
       [0, 0, 0, 0, 0, 8]])

In [4]:
basis_map_frac=ainv@s@basis_map
basis_map_frac

array([[ 1.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 4.   ,  0.   ,  0.   ,  0.   ,  3.   ,  1.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  1.   ,  0.   ,  0.   ],
       [-1.   ,  0.   ,  0.   , -0.75 , -0.75 , -0.25 ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.25 ,  0.   ,  0.   , -0.25 ,  0.   ,  0.   ],
       [-1.   ,  0.   ,  0.75 ,  0.   ,  0.25 ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.25 ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [-0.5  ,  0.125,  0.875,  0.875,  1.125,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.25 ,  0.   ,  0.   ,  0.   ,  0.25 ,  0.   ,  0.   ,  0.25 ,  0.   ,  0.   ],
       [ 0.   , -0.25 , -1.   , -1.   , -1.5  ,  0.125,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.25 ,  0.   ,  0.

In [6]:
basis_map_frac@c2_vec

array([ 48., 246., -64., -48., -24.,   8.])

In [7]:
ainv

array([[1.   , 0.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 1.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.25 , 0.   , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.25 , 0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.125, 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.125]])

In [8]:
# cy_cycles=np.zeros((len(two_cones),len(four_cones)),dtype=int)
# for i,c2 in enumerate(two_cones):
#     for j,c4 in enumerate(four_cones):
#         if np.all([x in c4 for x in c2]):
#             dif = sorted([x for x in c4 if not x in c2])
#             if dif in combs:
#                 cy_cycles[i,j]=1
# # cy_cycles=np.rint(cy_cycles).astype(int)

In [14]:
V=np.array(fan.vectors()); nrays=len(V)
LB1,LB2=F.nef_partition_M(as_divisors=True)
def move_off(L,tau):
    mc=next(c for c in fan.cones() if set(tau)<=set(c))
    rhs=np.array([-L[r-1] if r in tau else 0 for r in mc])
    mm=np.rint(np.linalg.solve(V[np.array(mc)-1],rhs)).astype(int)
    return L+V@mm

cy_cycles=np.zeros((len(two_cones),len(four_cones)),dtype=int)
for i,c2 in enumerate(two_cones):
    L1=move_off(LB1,c2)                               
    for j,c4 in enumerate(four_cones):
        if np.all([x in c4 for x in c2]):
            x,y=[r for r in c4 if r not in c2]
            cy_cycles[i,j]=L1[x-1]*move_off(LB2,c2+(x,))[y-1]+L1[y-1]*move_off(LB2,c2+(y,))[x-1]
B=cy_cycles@m.T
rankB=np.linalg.matrix_rank(B)
print(UF.snf(B)[:rankB,:rankB])

[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 4 0 0 0]
 [0 0 0 4 0 0]
 [0 0 0 0 8 0]
 [0 0 0 0 0 8]]


In [20]:
# --------------------------------------------------
# 1. CY intersection matrix on raw two-cone generators
# --------------------------------------------------

K = cy_cycles @ m.T

hK, tK = UF.hnf(K, return_transform=True)

nz = np.flatnonzero(np.any(hK != 0, axis=1))
rK = len(nz)

assert rK == 6

# 6 x 98:
# each row is a combination of the raw two-cones
Qcy = tK[nz, :]

# Check that the other transformed generators are null
zrows = np.flatnonzero(~np.any(hK != 0, axis=1))
assert np.all(tK[zrows, :] @ K == 0)

# Gram matrix in this Z-basis
Gcy = Qcy @ K @ Qcy.T

print("Gcy =")
print(Gcy)
print("SNF(Gcy) =")
print(UF.snf(Gcy))

Gcy =
[[ 8  4  8  2  8 12]
 [ 4 -4  4  1  0  4]
 [ 8  4  4  1  8 12]
 [ 2  1  1  0  2  3]
 [ 8  0  8  2  0  8]
 [12  4 12  3  8 12]]
SNF(Gcy) =
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 4 0 0 0]
 [0 0 0 4 0 0]
 [0 0 0 0 8 0]
 [0 0 0 0 0 8]]


In [21]:
# --------------------------------------------------
# 2. Integral basis of ambient numerical 4-cycle quotient
# --------------------------------------------------

b4, t4 = UF.hnf(m.T, return_transform=True)

nz4 = np.flatnonzero(np.any(b4 != 0, axis=1))
s = len(nz4)

print("ambient rank =", s)

# Since t4 is unimodular
t4inv0 = np.linalg.inv(t4)

if not np.allclose(t4inv0, np.rint(t4inv0), atol=1e-10):
    raise ValueError("Numerical inverse of t4 is not reliably integral")

t4inv = np.rint(t4inv0).astype(int)

assert np.array_equal(
    t4 @ t4inv,
    np.eye(t4.shape[0], dtype=int)
)

# Coordinates of pushforward of every raw two-cone class
# in the ambient numerical quotient basis
E_all_full = cy_cycles @ t4inv

# If HNF puts nonzero rows first:
E_all = E_all_full[:, nz4]

print("E_all shape =", E_all.shape)

ambient rank = 23
E_all shape = (98, 23)


In [22]:
# --------------------------------------------------
# 3. Pushforward of the six CY basis classes
# --------------------------------------------------

E = Qcy @ E_all

print("E shape =", E.shape)
print(E)

SE = UF.snf(E)

print("SNF of pushforward embedding:")
print(SE)

E shape = (6, 23)
[[  0   4   0   0   0   0   0   0   0   0   0   0   0   2   0   0   0   8   0   0   0  16  12]
 [  0   0   0   8   0   0   0  32   0   0   0   0   0   1   0   0   0   0   0   0   0 -16   4]
 [  0   0   0   0   0   4   0   0   0   0   0   0   0   1   0   0   0   8   0   0   0  16  12]
 [  0   0   0   0   0   0   4   0   1   0   0   0   0   0   0   0   0   2   0   0   0   2   3]
 [  0   0   0   0   0   0   0   0   0   0   0   8   0   2   0   0  32   0   0   0   0   0   8]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   3   0   0   0   8   0   0   0  16  12]]
SNF of pushforward embedding:
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 4 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 4 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 8 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 8 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [10]:
F.nef_partition_M(as_divisors=True)

[array([0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 array([1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0])]

In [59]:
UF.snf(B)[:rankB,:rankB]

array([[1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 4, 0, 0, 0],
       [0, 0, 0, 4, 0, 0],
       [0, 0, 0, 0, 8, 0],
       [0, 0, 0, 0, 0, 8]])

In [38]:
np.gcd.reduce(B,axis=1)

array([ 0,  4,  0,  8,  0,  4,  4, 32,  1,  0,  0,  8,  0,  1,  0,  0, 32,  2,  0,  0,  0,  8,  0,  8,  1,  0,  0,  4,  0,  0,  8,  4,  4,  0,  0,  4,  0,  0,  0,  1,  0,  0,  4,  0,  4,  1,  0,  0,  0,  1,  1,  0,  0,  0,  0,  8,  4,  2,  0,  0,  4,  1,  2,  0,  0,  0,  4,  8,  0,  0,  4,  8,  4,  1,  8,  1,  8,  8,  4,  1,  0,  0,  0,  0,  1,  0,  0,  0,  1,  0,  0,  1,  0,  2,  0,  4,  0,  8])

In [13]:
cy_cycles[0]

array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  

In [23]:
H4coords(cy_cycles).shape

(98, 23)

In [25]:
C = H4coords(cy_cycles)              # 98 x 23 : each CY surface as an integral class in H_4(V6,Z) ≅ Z^23
A, S, T = UF.snf(C, return_decomposition=True)      # A = S @ C @ T  (diagonal)
d = [int(A[i,i]) for i in range(min(A.shape)) if A[i,i] != 0]     # [1, 1, 4, 4, 8, 8]
f = np.rint(np.linalg.inv(T)).astype(int)            # rows: a new Z-basis f_1..f_23 of H_4(V6,Z)


In [26]:
f

array([[  0,   0,   0,   0,   0,   0,   4,   0,   1,   0,   0,   0,   0,   0,   0,   0,   0,   2,   0,   0,   0,   2,   3],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  -8,   0,   1,   0,   0, -32,   8,   0,   0,   0,  16,   4],
       [  0,   1,   0,   0,   0,  -2,   0,   0,   0,   0,   0,  -8,   0,   1,   0,   0, -32,   6,   0,   0,   0,  12,   1],
       [  0,   0,   0,   0,   0,   1, -12,   0,  -3,   0,   0,  -8,   0,   2,   0,   0, -32,   6,   0,   0,   0,  18,   1],
       [  0,   0,   0,  -1,   0,   1,  -8,  -4,  -2,   0,   0,   2,   0,   1,   0,   0,   8,  -1,   0,   0,   0,   4,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  -3,   0,   0,   0,   0, -12,   2,   0,   0,   0,   4,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,   0],
       [

In [15]:
np.linalg.matrix_rank(B)

6

In [18]:
UF.snf(B)[:6,:6]

array([[1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 4, 0, 0, 0],
       [0, 0, 0, 4, 0, 0],
       [0, 0, 0, 0, 8, 0],
       [0, 0, 0, 0, 0, 8]])

In [121]:
cy_cycles.shape

(98, 368)

In [124]:
(M@cy_cycles.T).shape

(23, 98)

In [126]:
(N@cy_cycles).shape

(23, 368)

In [66]:
t4@m.T==b4

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]])

In [61]:
basis4_cycles[0]

array([ 1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  3,  0,  0,  0,  0,  0, -2,  3,  0,  0,  0,  0,  0,  0,  2,  1,  0,  0,  3,  0,  0,  0,  1,  0,  1, -1,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  2,  0,  0,  0,  0, -3,  0,  0,  0,  1, -3,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  1,  0, -3,  0,  0,  0,  0, -1,  0,  0, -1,  0,  0,  0,  0,  0,  0,  0])

In [21]:
from itertools import combinations
V=np.array(fan.vectors()); nrays=len(V)
cones={tuple(sorted(t)) for c in fan.cones() for k in range(1,7) for t in combinations(c,k)}
col={tuple(sorted(c)):j for j,c in enumerate(four_cones)}
LB1=np.isin(np.arange(1,nrays+1),np1).astype(int); LB2=1-LB1

def ray(i):
    e=np.zeros(nrays,int); e[i-1]=1; return e

def move_off(L,tau):
    """Divisor linearly equivalent to L with zero coefficients on the rays of tau."""
    mc=next(c for c in fan.cones() if set(tau)<=set(c))
    rhs=np.array([-L[r-1] if r in tau else 0 for r in mc])
    mm=np.rint(np.linalg.solve(V[np.array(mc)-1],rhs)).astype(int)   # smooth: unimodular
    return L+V@mm

def cycle(tau,Ls):
    """V(tau).L_1...L_k expanded into 4-cone classes (vector in Z^{#four_cones})."""
    out=np.zeros(len(four_cones),int)
    if not Ls:
        out[col[tuple(sorted(tau))]]=1; return out
    L=move_off(Ls[0],tau)
    for r in np.nonzero(L)[0]+1:
        t=tuple(sorted(tau+(r,)))
        if t in cones: out+=L[r-1]*cycle(t,Ls[1:])
    return out

def H4coords(rows):
    """Quotient by the relations: coordinates in the Z-basis basis4_cycles of H_4(A,Z)."""
    P=np.asarray(rows)@m.T
    C=np.linalg.lstsq(basis4_cycles.T.astype(float),P.T.astype(float),rcond=None)[0].T
    Ci=np.rint(C).astype(int); assert np.allclose(C,Ci); return Ci

def elem_divs(C):
    d=np.diag(UF.snf(C)); return [int(x) for x in d if x!=0]


In [134]:
# CY surfaces D_a D_b L1 L2 -> your SNF, now in H_4(A,Z)
C_cy=H4coords([cycle(c2,[LB1,LB2]) for c2 in two_cones])
print(elem_divs(C_cy))                                   # [1, 1, 4, 4, 8, 8]

# extra integral cycles of X: fibral divisors over 2-cycles of S (resp. of the double cover)
H=ray(3)                                                 # pi^* of the hyperplane of P^3
long_=H4coords([cycle((f,),[H,H,LB2]) for f in (1,9,11)])
short=H4coords([cycle((f,),[H,H,LB2]) for f in (10,12)]); assert (short%2==0).all()
C_X=np.vstack([C_cy,long_,short//2])
print(elem_divs(C_X))                                    # all 1: saturated in H_4(A,Z)

Bx=UF.hnf(C_X); Bx=Bx[Bx.any(1)]
K=np.rint(np.linalg.lstsq(Bx.T.astype(float),C_cy.T.astype(float),rcond=None)[0].T).astype(int)
assert np.array_equal(K@Bx,C_cy)
print(elem_divs(K), np.prod(elem_divs(K)))               # [1, 1, 4, 4, 8, 8] 1024


[1, 1, 4, 4, 8, 8]
[1, 1, 1, 1, 1, 1]
[1, 1, 4, 4, 8, 8] 1024


In [ ]:
"""
Pushforward of CICY double intersections D_a D_b . Theta_B Theta_W into the integral Chow
group A_{n-4}(V) of the toric ambient V (generated by the 4-cone strata), saturation of
their span and its index.  Exact integer/Fraction arithmetic, numpy object arrays, no sympy.

CYTools input:
    t = Polytope(pts).triangulate(make_star=True); pts = t.points()
    o = next(i for i,p in enumerate(pts) if not any(p)); keep = [j for j in range(len(pts)) if j != o]
    rays = [pts[j] for j in keep]; idx = {j:k for k,j in enumerate(keep)}
    cones = [[idx[i] for i in s if i != o] for s in t.simplices()]
"""
import itertools, numpy as np
from fractions import Fraction as Fr


def snf(A):
    """Smith normal form over Z: d, U, V with U @ A @ V = diag(d), Python-int object arrays."""
    A = np.array([[int(x) for x in row] for row in A], dtype=object); m, n = A.shape
    U, V = np.eye(m, dtype=object), np.eye(n, dtype=object)
    for t in range(min(m, n)):
        while True:
            nz = [(abs(A[i, j]), i, j) for i in range(t, m) for j in range(t, n) if A[i, j]]
            if not nz: return [A[i, i] for i in range(t)] + [0] * (min(m, n) - t), U, V
            _, i, j = min(nz)
            A[[t, i]], U[[t, i]] = A[[i, t]], U[[i, t]]
            A[:, [t, j]], V[:, [t, j]] = A[:, [j, t]], V[:, [j, t]]
            for i in range(t + 1, m):
                q = A[i, t] // A[t, t]; A[i] -= q * A[t]; U[i] -= q * U[t]
            for j in range(t + 1, n):
                q = A[t, j] // A[t, t]; A[:, j] -= q * A[:, t]; V[:, j] -= q * V[:, t]
            if any(A[i, t] for i in range(t + 1, m)) or any(A[t, j] for j in range(t + 1, n)): continue
            bad = [i for i in range(t + 1, m) for j in range(t + 1, n) if A[i, j] % A[t, t]]
            if not bad: break
            A[t] += A[bad[0]]; U[t] += U[bad[0]]
    return [A[i, i] for i in range(min(m, n))], U, V


def solve(A, b):
    """rational solution of A m = b via SNF (integral whenever one exists); None if inconsistent."""
    d, U, V = snf(A); c = U @ np.array(b, dtype=object); y = np.zeros(V.shape[0], dtype=object)
    for j, cj in enumerate(c):
        if j < len(d) and d[j]: y[j] = F(int(cj), int(d[j]))
        elif cj: return None
    return V @ y


class ToricChow:
    def __init__(self, rays, max_cones):
        self.rays = np.array(rays, dtype=object); self.N, self.n = self.rays.shape
        self.cones = {frozenset(s) for c in max_cones for k in range(len(c) + 1) for s in itertools.combinations(c, k)}
        self.m = {}
    def mult(self, s):
        if s not in self.m:
            d = snf(self.rays[sorted(s)])[0] if s else [1]
            self.m[s] = abs(int(np.prod([x for x in d if x])))
        return self.m[s]
    def mul(self, cls, D):
        """multiply a class {cone: coeff} by the divisor with prime-divisor coefficients D."""
        out = {}
        for i, a in enumerate(D):
            if not a: continue
            for s, c in cls.items():
                if i not in s:
                    t = s | {i}
                    if t in self.cones: out[t] = out.get(t, 0) + a * c * F(self.mult(s), self.mult(t))
                else:
                    m = solve(self.rays[sorted(s)], [int(j == i) for j in sorted(s)])
                    for r in range(self.N):
                        t = s | {r}
                        if r not in s and t in self.cones and (p := m @ self.rays[r]):
                            out[t] = out.get(t, 0) - a * c * p * F(self.mult(s), self.mult(t))
        return {s: c for s, c in out.items() if c}
    def product(self, divisors):
        cls = {frozenset(): F(1)}
        for D in divisors: cls = self.mul(cls, D)
        return cls
    def degree(self, cls):
        return sum(c / self.mult(s) for s, c in cls.items() if len(s) == self.n)
    def chow(self, k):
        """generators (k-cones) and Fulton-Sturmfels relations of A_{n-k}(V)."""
        gens = sorted((s for s in self.cones if len(s) == k), key=sorted); pos = {s: j for j, s in enumerate(gens)}
        rels = []
        for tau in (s for s in self.cones if len(s) == k - 1):
            d, U, V = snf(self.rays[sorted(tau)]) if tau else ([], None, np.eye(self.n, dtype=object))
            for m in V[:, [j for j in range(self.n) if j >= len(d) or not d[j]]].T:   # Z-basis of tau^perp
                row = np.zeros(len(gens), dtype=object)
                for r in range(self.N):
                    sig = tau | {r}
                    if r not in tau and sig in self.cones: row[pos[sig]] += (m @ self.rays[r]) * F(self.mult(tau), self.mult(sig))
                if row.any(): rels.append(row)
        return gens, pos, np.array(rels, dtype=object)


def pushforward_lattice(rays, max_cones, thetaB, thetaW, divisors=None, coeff=False, verbose=True):
    """
    thetaB, thetaW: lists of ray indices (nef partition), or coefficient vectors if coeff=True.
    divisors: coefficient vectors of the H_a on the prime divisors (default: all prime divisors).
    Computes the lattice spanned by i_*(S_ab) in A_{n-4}(V), its saturation and the index.
    Since i_* PD(i^*alpha) = PD(alpha . Theta_B Theta_W), a rational combination of the S_ab pushes
    forward integrally iff it pairs integrally with all S_kl: the saturation is i_*(L^vee), the index
    is |disc L|.  The true lattice H^4(Y,Z) ∩ (L⊗Q) is not visible in this computation.
    """
    X = ToricChow(rays, max_cones); N, n = X.N, X.n
    vec = (lambda T: np.array(T, dtype=object)) if coeff else (lambda T: np.array([int(i in T) for i in range(N)], dtype=object))
    DB, DW = vec(thetaB), vec(thetaW)
    divisors = np.eye(N, dtype=object) if divisors is None else np.array(divisors, dtype=object)
    bad = sum(X.mult(s) != 1 for s in X.cones if len(s) == 4)
    gens, pos, R = X.chow(4)
    lcm = lambda xs: int(np.lcm.reduce([int(x) for x in xs] or [1]))
    den = lcm(x.denominator for x in R.flat)
    d, U, V = snf(R * den) if len(R) else ([], None, np.eye(len(gens), dtype=object))
    r = sum(1 for x in d if x); tors = [int(x) for x in d[:r] if abs(x) != 1]
    pairs = list(itertools.combinations_with_replacement(range(len(divisors)), 2))
    push = []
    for a, b in pairs:
        v = np.zeros(len(gens), dtype=object)
        for s, c in X.product([divisors[a], divisors[b], DB, DW]).items(): v[pos[s]] = c
        push.append((v @ V)[r:])                                           # free coordinates of A_{n-4}
    push = np.array(push, dtype=object)
    G = np.array([[X.degree(X.product([divisors[a], divisors[b], divisors[c], divisors[e], DB, DW])) for c, e in pairs] for a, b in pairs], dtype=object)
    pden = lcm(x.denominator for x in push.flat); dp, Up, _ = snf(push * pden)
    inv = [abs(int(x)) for x in dp if x]; gden = lcm(x.denominator for x in G.flat); dG = snf(G * gden)[0]
    sat = [(c, c @ G @ c) for c in (Up[j] / F(inv[j]) for j in range(len(inv)))]   # saturation generators as combos of S_ab
    if verbose:
        print(f"dim V = {n}, {N} rays, non-unimodular 4-cones: {bad}; rank A_{{n-4}}(V) = {len(gens) - r}, torsion {tors}")
        print(f"pushforward: rank {len(inv)}, SNF invariants {inv}, index of i_*(L) in saturation = {int(np.prod(inv))}" + ("  (pushforwards had denominators!)" if pden != 1 else ""))
        print(f"CY Gram matrix: integral={gden == 1}, rank {sum(1 for x in dG if x)}, |disc L| = {abs(int(np.prod([x for x in dG if x])))}")
        for c, si in sat:
            print("   saturation generator", {pairs[i]: str(c[i]) for i in range(len(pairs)) if c[i]}, " self-intersection", si)
    return dict(index=int(np.prod(inv)), invariants=inv, gram=G, pairs=pairs, saturation=sat, rank_ambient=len(gens) - r, torsion=tors, bad_cones=bad)

In [43]:
"""Mirror-octic uplift (paper §4): double CY intersections inside the ambient quadruple intersections."""
import itertools, numpy as np
from fractions import Fraction as Fr
from cytools import Polytope
# from pushforward_lattice import ToricChow, pushforward_lattice, snf

rays = np.array([[1,1,1,1,-3,0,0,0,-2,-2,-1,-1],      # columns of (4.6): code index i = Theta_{i+1}
                 [-1,0,0,1,0,0,0,0,0,0,0,0],
                 [-1,0,1,0,0,0,0,0,0,0,0,0],
                 [-1,1,0,0,0,0,0,0,0,0,0,0],
                 [0,0,0,0,-2,-2,0,1,-2,-1,-2,0],
                 [0,0,0,0,5,5,-1,-1,5,3,5,1]]).T
thetaB, thetaW = [0,1,2,3], list(range(4,12))
J = {'B':[0], '2':[4,8,10,5], '3':[8,10,5], '4':[10,5], '5':[5], 't':[7]}   # J^a as sums of Theta's
names = list(J); divisors = [[int(i in J[a]) for i in range(12)] for a in names]
target = {('B','B','B','t'):2, ('B','B','B','2'):1, ('B','B','2','2'):-4, ('B','2','2','2'):16, ('2','2','2','2'):-64}

# --- triangulation: pick the fine regular star triangulation of these 12 points that reproduces (4.20)
p = Polytope(rays.tolist() + [[0]*6]); pts = p.points()
o = next(i for i,q in enumerate(pts) if not any(q)); keep = [j for j in range(len(pts)) if j != o]
ray_index = {tuple(pts[j]): k for k, j in enumerate(keep)}; order = [ray_index[tuple(r)] for r in rays]  # map to paper order
# for t in p.all_triangulations(only_fine=True, only_regular=True, only_star=True):
t0 = p.triangulate(points=rays, make_star=True)
for t in [t0] + list(t0.neighbor_triangulations()):
    cones = [[order.index(ray_index[tuple(pts[i])]) for i in s if i != o] for s in t.simplices()]
    X = ToricChow(rays, cones); DB, DW = [int(i in thetaB) for i in range(12)], [int(i in thetaW) for i in range(12)]
    kap = lambda *a: X.degree(X.product([divisors[names.index(x)] for x in a] + [DB, DW]))
    if all(kap(*k) == v for k, v in target.items()): break
else: raise RuntimeError("no triangulation matches (4.20)")

# --- 1. expansion of each S_ab = J^a J^b Theta_B Theta_W in the 4-cone surfaces of the ambient space
for a, b in itertools.combinations_with_replacement(range(6), 2):
    cls = X.product([divisors[a], divisors[b], DB, DW])
    print(f"S_{names[a]}{names[b]} =", " + ".join(f"({c})V{tuple(i+1 for i in sorted(s))}" for s, c in cls.items()) or "0")

# --- 2. lattice they span in A_2(V), its saturation, the index, and the saturation generators
res = pushforward_lattice(rays, cones, thetaB, thetaW, divisors=divisors)
G = res['gram']; print("SNF of CY Gram matrix:", [int(x) for x in snf(G)[0]])

TypeError: cannot use 'numpy.ndarray' as a set element (unhashable type: 'numpy.ndarray')

In [50]:
2**(5)

32